# Text-to-Speech AI Inference — API Demo

**English / Tiếng Việt** · HTTP-only demo for Colab, Kaggle and Jupyter. No deployment or GPU startup.

**EN:** Set an existing gateway URL and a client key before running. Model audio is generated only when you deliberately submit a job.

**VI:** Cấu hình URL gateway đã triển khai và token client trước khi chạy. Chỉ gửi job nếu bạn muốn chủ động sử dụng GPU.


In [ ]:
import os
import time
import json
import getpass
from urllib.parse import urlparse
import uuid
import requests

TEXT_TO_SPEECH_GATEWAY_URL = os.getenv("TEXT_TO_SPEECH_API_BASE_URL", "").rstrip("/") or input("Existing TTS gateway URL (https://...): ").rstrip("/")
parsed_gateway_url = urlparse(TEXT_TO_SPEECH_GATEWAY_URL)
if parsed_gateway_url.scheme != "https" and not (parsed_gateway_url.scheme == "http" and parsed_gateway_url.hostname in {"localhost", "127.0.0.1", "::1"}):
    raise ValueError("Use HTTPS for a remote server (HTTP only for local CPU mock)")
CLIENT_BEARER_TOKEN = os.getenv("TEXT_TO_SPEECH_API_BEARER_TOKEN", "") or getpass.getpass("Bearer token: ")
AUTHENTICATED_REQUEST_HEADERS = {"Authorization": f"Bearer {CLIENT_BEARER_TOKEN}"}
print("Gateway URL configured; token is never printed")


## Discover options / Khám phá tùy chọn

**EN:** The model list includes a JSON Schema for the exact options each backend supports.

**VI:** Danh sách model trả về JSON Schema, giúp tạo form riêng cho từng backend mà không hard-code.


In [ ]:
model_catalog_response = requests.get(f"{TEXT_TO_SPEECH_GATEWAY_URL}/v1/models", headers=AUTHENTICATED_REQUEST_HEADERS, timeout=30)
model_catalog_response.raise_for_status()
for model in model_catalog_response.json()["data"]:
    print(model["id"], "→", model["checkpoint"])
    print(" Controls:", ", ".join(model["model_config_schema"]["properties"].keys()))


## Submit one job / Gửi một job

**EN:** Select a model and edit its options. The example is short to bound GPU use; this cell starts a paid remote inference job.

**VI:** Chọn model và sửa các tham số. Ví dụ ngắn nhằm giới hạn chi phí GPU; cell này thực sự gửi job tới GPU từ xa.


In [ ]:
SELECTED_MODEL_ID = "voxcpm2"
TEXT_TO_SYNTHESIZE = "Xin chào! Đây là một ví dụ tổng hợp giọng nói bằng trí tuệ nhân tạo."
MODEL_INFERENCE_OPTIONS = {
    "gender": "female", "age": "young",
    "emotion": "cheerful", "pace": "slow",
    "cfg_value": 2.0, "inference_timesteps": 10,
}
# Alternatives:
# SELECTED_MODEL_ID = "moss-tts-v1.5"; MODEL_INFERENCE_OPTIONS = {"language": "Vietnamese", "audio_temperature": 1.7}
# SELECTED_MODEL_ID = "higgs-tts-3"; MODEL_INFERENCE_OPTIONS = {"emotion": "enthusiasm", "speed": "slow"}

# Preserve STABLE_REQUEST_ID on notebook cell reruns. Reuse it after a lost POST ACK.
STABLE_REQUEST_ID = globals().get("STABLE_REQUEST_ID") or "nb-" + uuid.uuid4().hex
print("Stable Idempotency-Key for this notebook run:", STABLE_REQUEST_ID)
# EN: Keep the ID when retrying a request with the SAME payload; never automatically submit a new one.
# VI: Giữ nguyên ID nếu phải gửi lại cùng payload; không tự tạo job GPU mới.
# Set SUBMIT_PAID_INFERENCE_JOB=True only after checking model, text, rights, quota and cost.
SUBMIT_PAID_INFERENCE_JOB = False
if SUBMIT_PAID_INFERENCE_JOB:
    submission_response = requests.post(f"{TEXT_TO_SPEECH_GATEWAY_URL}/v1/audio/jobs", headers={**AUTHENTICATED_REQUEST_HEADERS, "Idempotency-Key": STABLE_REQUEST_ID, "Content-Type": "application/json"},
                         json={"model": SELECTED_MODEL_ID, "input": TEXT_TO_SYNTHESIZE, "model_config": MODEL_INFERENCE_OPTIONS}, timeout=45)
    submission_response.raise_for_status()
    SUBMITTED_JOB_ID = submission_response.json()["job_id"]
    print("Job:", SUBMITTED_JOB_ID, "reused:", submission_response.json().get("reused", False))
else:
    print("Submission disabled by default; set SUBMIT_PAID_INFERENCE_JOB=True to run real inference")


## Poll and play / Lấy trạng thái và nghe kết quả

**EN:** Poll only the job you submitted. The notebook does not automatically play or save any reference voice recordings.

**VI:** Chỉ kiểm tra job do bạn gửi. Notebook không tự tải hoặc phát tán file giọng tham chiếu.


In [ ]:
if SUBMIT_PAID_INFERENCE_JOB:
    for _ in range(60):
        job_status_response = requests.get(f"{TEXT_TO_SPEECH_GATEWAY_URL}/v1/audio/jobs/{SUBMITTED_JOB_ID}", headers=AUTHENTICATED_REQUEST_HEADERS, timeout=30)
        job_status_response.raise_for_status()
        job_status_data = job_status_response.json()
        print("State:", job_status_data["status"])
        if job_status_data["status"] in ("completed", "failed", "indeterminate"):
            break
        time.sleep(5)
    if job_status_data["status"] == "indeterminate":
        print("GPU outcome uncertain / Kết quả GPU chưa rõ; do NOT automatically retry")
    if job_status_data["status"] == "completed":
        audio_download_response = requests.get(f"{TEXT_TO_SPEECH_GATEWAY_URL}{job_status_data['audio_url']}", headers=AUTHENTICATED_REQUEST_HEADERS, timeout=120)
        audio_download_response.raise_for_status()
        from IPython.display import Audio, display
        display(Audio(data=audio_download_response.content, autoplay=False))
        print("Audio metadata:", job_status_data.get("audio"))
else:
    print("No job was sent")


## Important / Lưu ý

**EN:** Vocal attributes are best-effort model conditioning, not guaranteed identities. Use voice references only with informed permission. Higgs TTS 3 has restricted rights for hosted/commercial use. CPU mock audio is silence and must not be mistaken for synthetic speech.

**VI:** Tuổi/giới tính/cảm xúc chỉ là điều kiện gợi ý cho model, không đảm bảo cố định giọng. Chỉ clone giọng khi được cho phép. Higgs TTS 3 có giới hạn giấy phép khi cung cấp dịch vụ hoặc sử dụng thương mại. Audio CPU mock chỉ là im lặng, không phải TTS thật.
